# Small Example in the Physical Domain
This is a small example on how to derive the matrices using Substructuring Primal and Dual Assemblies. Substructure A is a three degree-of freedom system clamped at its left end and substructure B is a two degree-of-freedom structure clamped at its right end. The systems are coupled in their free ends.

2026-01-12
Andreas Linderholt, LNU

In [2]:
import numpy as np

## Initialize

In [3]:
m1 = 1; m2 = 1; m3 = 1; m4 = 1; m5 = 1;
c1 = 1; c2 = 2; c3 = 3; c4 = 4; c5 = 5;
k1 = 10000; k2 = 20000; k3 = 30000; k4 = 40000; k5 = 50000;

## Substructure A

In [4]:
MA = np.array([[m1, 0,  0],
               [0,  m2, 0],
               [0,  0,  m3]])

CA = np.array([[(c1+c2), -c2,      0],
               [-c2,     (c2+c3), -c3],
               [ 0,       -c3,      c3]])

KA = np.array([[(k1+k2), -k2,      0],
               [-k2,     (k2+k3), -k3],
               [ 0,      -k3,      k3]])

FA = np.array([1, 0, 0])[..., np.newaxis] # np.newaxis makes this a column vector
GA = np.array(['g1', 'g2', 'g3'])[..., np.newaxis] # np.newaxis makes this a column vector
rsubA, csubA = MA.shape

## Substructure B

In [5]:
MB = np.array([[m4, 0],
               [0,  m5]])

CB = np.array([[ c4, -c4],
               [-c4, (c4+c5)]])

KB = np.array([[ k4, -k4],
               [-k4, (k4+k5)]])

FB = np.array([0, 0])[...,np.newaxis] # np.newaxis makes this a column vector
GB = np.array(['g4', 'g5'])[...,np.newaxis] # np.newaxis makes this a column vector
rsubB, csubB = MB.shape

## Block Diagonal Matrices and Vectors

In [6]:
M = np.vstack((np.column_stack((MA, np.zeros((rsubA,csubB)))),
               np.column_stack((np.zeros((rsubB,csubA)), MB))))

C = np.vstack((np.column_stack((CA, np.zeros((rsubA,csubB)))),
               np.column_stack((np.zeros((rsubB,csubA)), CB))))

K = np.vstack((np.column_stack((KA, np.zeros((rsubA,csubB)))),
               np.column_stack((np.zeros((rsubB,csubA)), KB))))

F = np.vstack((FA, FB))

G = np.vstack((GA, GB))

## The Compatibility Matrix, B

In [7]:
B = np.array([0, 0, 1, -1, 0])[np.newaxis,...] # np.newaxis makes this a row vector
rB, cB = B.shape

## The Localization Matrix, L

In [ ]:
L = np.array([[1, 0, 0, 0],
              [0, 1, 0, 0],
              [0, 0, 1, 0],
              [0, 0, 1, 0],
              [0, 0, 0, 1]])

## The Three Field Formulation
$$
\overline{M}\ddot{x} + \overline{C}\dot{x} + \overline{K}x = \overline{F}+\overline{G} \tag{1: Governing equation of motion}
$$
$$
Bx = 0 \tag{2: Compatibility}
$$
$$
L^TG=0 \tag{3: Equilibrium}
$$

## Primal Assembly
$$
x = Lx_{global} \tag{4}
$$
Substituting $(4)$ into $(1)$ and pre-multiply with $L^T$ results in:
$$
L^T\overline{M}L\ddot{x}_{global} + L^T\overline{C}L\dot{x}_{global} + L^T\overline{K}Lx_{global} = L^T\overline{F} + L^T\overline{G} 
$$
Using $(3)$:
$$
L^T\overline{M}L\ddot{x}_{global} + L^T\overline{C}L\dot{x}_{global} + L^T\overline{K}Lx_{global} = L^T\overline{F}
$$

In [9]:
Mprimal = L.transpose()@M@L
print('Mprimal:')
print(Mprimal)

Mprimal:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 2. 0.]
 [0. 0. 0. 1.]]


In [10]:
Cprimal = L.transpose()@C@L
print('Cprimal:')
print(Cprimal)


Cprimal:
[[ 3. -2.  0.  0.]
 [-2.  5. -3.  0.]
 [ 0. -3.  7. -4.]
 [ 0.  0. -4.  9.]]


In [11]:
Kprimal = L.transpose()@K@L
print('Kprimal:')
print(Kprimal)

Kprimal:
[[ 30000. -20000.      0.      0.]
 [-20000.  50000. -30000.      0.]
 [     0. -30000.  70000. -40000.]
 [     0.      0. -40000.  90000.]]


In [12]:
Fprimal = L.transpose()@F
print('Fprimal:')
print(Fprimal)

Fprimal:
[[1]
 [0]
 [0]
 [0]]


## Dual Assembly
Inserting $G=-B^T\lambda$ into $(3)$:

$$
L^T\begin{pmatrix}-B^T\lambda\end{pmatrix} = -L^TB^T\lambda = -\begin{pmatrix}BL\end{pmatrix}^T\lambda = 0
$$

$(3)$ is, by the choice of $G$, fulfilled and:

$$
\overline{M}\ddot{x} + \overline{C}\dot{x} + \overline{K}x = \overline{F} - B^T\lambda
$$

Which simplifies to:

$$
\overline{M}\ddot{x} + \overline{C}\dot{x} + \overline{K}x + B^T\lambda = \overline{F} 
$$

With the displacement vector $[x;\lambda]$:

$$
\begin{bmatrix}\overline{M} & 0 \\ 0 & 0 \end{bmatrix} \begin{Bmatrix} \ddot{x} \\ \ddot{\lambda} \end{Bmatrix} + \begin{bmatrix}\overline{C} & 0 \\ 0 & 0 \end{bmatrix} \begin{Bmatrix} \dot{x} \\ \dot{\lambda} \end{Bmatrix} + \begin{bmatrix}\overline{K} & B^T \\ B & 0 \end{bmatrix} \begin{Bmatrix} x \\ \lambda \end{Bmatrix} = \begin{Bmatrix} \overline{F} \\ 0 \end{Bmatrix}
$$

In [13]:
Mdual = np.vstack((np.column_stack((M, np.zeros((cB,rB)))),
                   np.column_stack((np.zeros((rB,cB)), np.zeros((rB,rB))))))
print('Mdual:')
print(Mdual)

Mdual:
[[1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0.]]


In [14]:
Cdual = np.vstack((np.column_stack((C, np.zeros((cB,rB)))),
                   np.column_stack((np.zeros((rB,cB)), np.zeros((rB,rB))))))
print('Cdual:')
print(Cdual)

Cdual:
[[ 3. -2.  0.  0.  0.  0.]
 [-2.  5. -3.  0.  0.  0.]
 [ 0. -3.  3.  0.  0.  0.]
 [ 0.  0.  0.  4. -4.  0.]
 [ 0.  0.  0. -4.  9.  0.]
 [ 0.  0.  0.  0.  0.  0.]]


In [ ]:
Kdual = np.vstack((np.column_stack((K, B.transpose())),
                   np.column_stack((B, np.zeros((rB,rB))))))
print('Kdual:')
print(Kdual.astype(int)) # Printed as integer to avoid scientific notation

Kdual:
[[ 30000 -20000      0      0      0      0]
 [-20000  50000 -30000      0      0      0]
 [     0 -30000  30000      0      0      1]
 [     0      0      0  40000 -40000     -1]
 [     0      0      0 -40000  90000      0]
 [     0      0      1     -1      0      0]]


In [19]:
Fdual=np.vstack((F, np.zeros((rB,1))))
print('Fdual:')
print(Fdual)

Fdual:
[[1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
